In [1]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="flan_t5_true_false_first_step_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [4]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

true_ids = tokenizer("true", add_special_tokens=False).input_ids
false_ids = tokenizer("false", add_special_tokens=False).input_ids

print("model:", model_name)
print("true token ids:", true_ids)
print("false token ids:", false_ids)

if len(true_ids) != 1 or len(false_ids) != 1:
    raise ValueError(
        f"Expected single-token verbalizers, got true={true_ids}, false={false_ids}"
    )

true_token_id = true_ids[0]
false_token_id = false_ids[0]

decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

print("decoder_start_token_id:", decoder_start_token_id)


---[ TableVault Record ]---


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model: google/flan-t5-small
true token ids: [1176]
false token ids: [6136]
decoder_start_token_id: 0
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

prompts = [
    f"Determine whether the two sentences are paraphrases. Answer true or false.\nSentence 1: {s1}\nSentence 2: {s2}"
    for s1, s2 in zip(sent1, sent2)
]

print("num_examples:", len(ds))
print("positive_rate:", y_true.mean())
print("example_prompt:\n", prompts[0])


---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
example_prompt:
 Determine whether the two sentences are paraphrases. Answer true or false.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
---[ TableVault Record ]---



In [6]:
batch_size = 32
preds = []
true_logits_all = []
false_logits_all = []

with torch.no_grad():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        bsz = enc["input_ids"].shape[0]
        decoder_input_ids = torch.full(
            (bsz, 1),
            decoder_start_token_id,
            dtype=torch.long,
            device=device,
        )

        outputs = model(**enc, decoder_input_ids=decoder_input_ids)
        first_step_logits = outputs.logits[:, 0, :]

        true_logits = first_step_logits[:, true_token_id]
        false_logits = first_step_logits[:, false_token_id]

        batch_preds = (true_logits > false_logits).long().cpu().numpy()
        preds.extend(batch_preds.tolist())
        true_logits_all.extend(true_logits.cpu().numpy().tolist())
        false_logits_all.extend(false_logits.cpu().numpy().tolist())

y_pred = np.array(preds)
true_logits_all = np.array(true_logits_all)
false_logits_all = np.array(false_logits_all)
print("done")


---[ TableVault Record ]---


  0%|          | 0/13 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [7]:

vault.create_record_list("google_flan_prediction_and_logits", column_names=["prediction", "true_logits", "false_logits"])

for i in range(len(y_pred)):
    vault.append_record("google_flan_prediction_and_logits", 
                        {
                            "prediction": int(y_pred[i]),
                            "true_logits": float(true_logits_all[i]),
                            "false_logits": float(false_logits_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example inference outputs from google/flan-t5-small on the GLUE MRPC validation set using a prompted true/false paraphrase decision. Each record corresponds to one validation example and is linked to the matching source row in glue_mrpc_validation. The dataset has three fields: prediction (binary model decision, where 1 means the logit for \u201ctrue\u201d is greater than the logit for \u201cfalse\u201d), true_logits (the first-decoder-step logit for the token \u201ctrue\u201d), and false_logits (the first-decoder-step logit for the token \u201cfalse\u201d). In this workflow, this dataset is the stored prediction artifact used to inspect model behavior on individual examples and to support the downstream summary evaluation dataset with accuracy, F1, and classification report."
embedding = get_embeddings(description)
vault.create_description("google_flan_prediction_and_logits", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions with logits", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prediction_format": "binary true/false", "inference_method": "first decoder-step logits comparison", "label_space": "0=not_paraphrase,1=paraphrase", "input_type": "sentence pair prompt", "output_columns": "prediction,true_logits,false_logits"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("google_flan_prediction_and_logits", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["not_paraphrase", "paraphrase"],
        )
print({"accuracy": acc, "f1": f1})
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["not_paraphrase", "paraphrase"],
    )
)


---[ TableVault Record ]---
{'accuracy': 0.5833333333333334, 'f1': 0.6443514644351465}
                precision    recall  f1-score   support

not_paraphrase       0.40      0.65      0.50       129
    paraphrase       0.77      0.55      0.64       279

      accuracy                           0.58       408
     macro avg       0.59      0.60      0.57       408
  weighted avg       0.66      0.58      0.60       408

---[ TableVault Record ]---



In [9]:
for i in range(5):
    print("=" * 80)
    print("idx:", i)
    print("prompt:\n", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", "true" if y_pred[i] == 1 else "false")
    print("true_logit:", float(true_logits_all[i]), "false_logit:", float(false_logits_all[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("prompt:\n", prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", "true" if y_pred[i] == 1 else "false")
    print("true_logit:", float(true_logits_all[i]), "false_logit:", float(false_logits_all[i]))


---[ TableVault Record ]---
idx: 0
prompt:
 Determine whether the two sentences are paraphrases. Answer true or false.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: true
true_logit: -4.720203399658203 false_logit: -5.303816318511963
idx: 1
prompt:
 Determine whether the two sentences are paraphrases. Answer true or false.
Sentence 1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
Sentence 2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: false
true_logit: -6.741540431976318 false_logit: -6.069624900817871
idx: 2
prompt:
 Determine whether the two sentences are paraphrases. Answer true or false.
Sentence 1: The dollar was at 116.92 yen against the yen , flat 

In [10]:

vault.create_record_list("flan_t5_true_false_first_step_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("flan_t5_true_false_first_step_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "google_flan_prediction_and_logits": [0, len(ds)]
                    })

summary

description = "Summary dataset for the FLAN-T5 MRPC evaluation run using first-step true/false token scoring. It contains aggregate performance results for google/flan-t5-small on the glue_mrpc_validation split, where each example is prompted as a paraphrase detection task and predicted by comparing the first decoder-step logits for the tokens true and false. The record list has three fields: accuracy (float), f1 (float, for the positive/paraphrase class), and classification_report (string containing the full per-class precision/recall/F1 summary). In this workflow, this dataset serves as the experiment-level evaluation artifact that summarizes model performance over the full validation set and links the source validation data with the per-example prediction/logit dataset google_flan_prediction_and_logits."
embedding = get_embeddings(description)
vault.create_description("flan_t5_true_false_first_step_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "evaluation summary", "source": "glue/mrpc", "input_dataset": "glue_mrpc_validation", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prediction_format": "true_false_first_step_logits", "metrics": "accuracy,f1,classification_report", "framework": "transformers"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_true_false_first_step_mrpc_summary", cat, embedding, prop)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [11]:
description = "This notebook evaluates google/flan-t5-small on the GLUE MRPC validation set for binary paraphrase detection using a prompt-based true/false formulation. It loads sentence pairs and labels from the glue_mrpc_validation dataset stored in TableVault, prompts the model with an instruction asking whether the two sentences are paraphrases, and uses the first decoder-step logits for the single-token verbalizers true and false to make predictions. The workflow records per-example predictions and logits in TableVault, computes summary metrics including accuracy, F1, and a classification report, inspects a few examples and errors, and stores dataset, result, and notebook-level descriptions with OpenAI text embeddings for metadata and retrieval." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("flan_t5_true_false_first_step_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "dataset": "glue/mrpc validation", "model": "google/flan-t5-small", "model_family": "flan-t5", "inference_type": "zero-shot text-to-text classification", "classification_method": "first-decoder-step true/false logit comparison", "prompt_format": "instruction prompt with sentence pair and true/false answer", "labels": "true=false paraphrase labels mapped to MRPC 1/0", "frameworks": "transformers, torch, datasets, scikit-learn", "evaluation": "accuracy, f1-score, classification_report", "device": "mps or cpu", "outputs": "per-example predictions and true/false logits, aggregate summary metrics", "storage": "tablevault with ArangoDB backend", "embedding_model": "text-embedding-3-large", "process_name": "flan_t5_true_false_first_step_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_true_false_first_step_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

